# 12 — YOLO OBB Card Extraction

Two-part notebook based on the approach from [1vcian/Pokemon-TCGP-Card-Scanner](https://github.com/1vcian/Pokemon-TCGP-Card-Scanner):

### Part A — Pre-trained model (Option 1: run immediately)
Download their YOLO11n-OBB model trained on 10k synthetic Pokemon card images,
run oriented bounding-box detection, perspective-warp the card to a clean crop,
then feed into our existing centering/defect detectors.

### Part B — Custom training dataset (Option 2: better accuracy)
Adapt `trainCreator.py` to composite our own card images onto diverse eBay-style
backgrounds, generate a YOLO OBB dataset, and train a fresh YOLO11n-obb model.

**Why OBB over axis-aligned boxes?**
OBB gives all 4 corners even when the card is tilted — required for a correct
perspective warp that produces a perfectly rectangular crop for grading analysis.

**Prerequisites:** `pip install ultralytics opencv-python-headless`

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
from pathlib import Path

# Test images for Part A
TEST_IMAGES = [
    "cards/image0_front.jpeg",
    "cards/image1_front.jpeg",
    "cards/image2_front.jpeg",
]

# Output card crop size (standard Pokemon card 63×88mm → 2.48:3.46 ratio)
CARD_W, CARD_H = 630, 880   # px — crop target for perspective warp

# Detection confidence threshold
CONF_THRESHOLD = 0.25

# Part B — training dataset paths
CARDS_DIR       = Path("datasets/psa_ebay_old")      # source card images
BACKGROUNDS_DIR = Path("datasets/backgrounds")        # background images (create if needed)
DATASET_OUT     = Path("datasets/yolo_obb_cards")     # generated YOLO dataset
N_TRAIN_IMAGES  = 2000    # synthetic images to generate (use 10000 for full training)

# Pre-trained model (Part A) — downloaded automatically by ultralytics
# This is the YOLO11n-obb base; replace with Hub model path once exported
PRETRAINED_BASE = "yolo11n-obb.pt"   # base model for transfer learning

# Their Ultralytics Hub model ID (requires API key for direct download)
HUB_MODEL_ID = "dQfecRsRsXbAKXOXHLHJ"
HUB_API_KEY  = ""   # ← paste your Ultralytics API key here if you have one

VERBOSE = True

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────────
import math, random, shutil, yaml, os
from io import BytesIO
from pathlib import Path
from typing import Optional

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from tqdm.notebook import tqdm
from ultralytics import YOLO

print("✅  Imports OK")

---
## Part A — Pre-trained Model
### A1 — Load model

In [ ]:
def load_obb_model() -> YOLO:
    """
    Load the best available OBB model:
      1. Their custom Hub model (needs API key) → best for Pokemon cards
      2. Cached custom model from previous training (Part B)
      3. YOLO11n-obb base model → general oriented-box detector

    The base model (option 3) is trained on DOTA (aerial objects) so it
    won't recognize "pokemon_card" as a class, but its OBB heads still detect
    rectangular objects. We use it for demo; train your own with Part B.
    """
    # Option 1: Hub model with API key
    if HUB_API_KEY:
        try:
            from ultralytics import hub
            hub.login(HUB_API_KEY)
            model = YOLO(f"https://hub.ultralytics.com/models/{HUB_MODEL_ID}")
            print(f"✅  Loaded Hub model {HUB_MODEL_ID}")
            return model
        except Exception as e:
            print(f"⚠️  Hub load failed: {e}")

    # Option 2: locally trained custom model from Part B
    custom_path = DATASET_OUT / "train" / "weights" / "best.pt"
    if custom_path.exists():
        model = YOLO(str(custom_path))
        print(f"✅  Loaded custom trained model: {custom_path}")
        return model

    # Option 3: base YOLO11n-obb (auto-downloaded ~6 MB)
    model = YOLO(PRETRAINED_BASE)
    print(f"✅  Loaded base {PRETRAINED_BASE} (DOTA pretrained — run Part B for card-specific weights)")
    return model


model = load_obb_model()

### A2 — OBB detection + perspective warp

The OBB result gives 4 corner points `(x1,y1)...(x4,y4)` in pixel coords.
We sort them into TL/TR/BR/BL order and apply `cv2.getPerspectiveTransform`
to warp the card to a clean `CARD_W × CARD_H` rectangle.

In [ ]:
def order_corners(pts: np.ndarray) -> np.ndarray:
    """
    Sort 4 corner points into [TL, TR, BR, BL] order.
    Works regardless of which corner the OBB starts from.
    """
    pts = pts.reshape(4, 2).astype(np.float32)
    # Top points have smallest y; bottom have largest
    s = pts.sum(axis=1)          # TL has smallest sum, BR has largest
    d = np.diff(pts, axis=1)     # TR has smallest diff, BL has largest
    ordered = np.zeros((4, 2), dtype=np.float32)
    ordered[0] = pts[np.argmin(s)]   # TL
    ordered[2] = pts[np.argmax(s)]   # BR
    ordered[1] = pts[np.argmin(d)]   # TR
    ordered[3] = pts[np.argmax(d)]   # BL
    return ordered


def perspective_warp(img: np.ndarray, corners: np.ndarray,
                     out_w: int = CARD_W, out_h: int = CARD_H) -> np.ndarray:
    """Perspective-warp the detected card region to a clean rectangle."""
    src = order_corners(corners)
    dst = np.array([[0, 0], [out_w, 0], [out_w, out_h], [0, out_h]],
                   dtype=np.float32)
    M = cv2.getPerspectiveTransform(src, dst)
    warped = cv2.warpPerspective(img, M, (out_w, out_h),
                                  flags=cv2.INTER_LANCZOS4)
    return warped


def detect_and_warp(image_path: str | Path,
                    conf: float = CONF_THRESHOLD) -> dict:
    """
    Run OBB detection on one image, return:
      - 'original'  : original BGR numpy array
      - 'crops'     : list of warped card crops (one per detection)
      - 'corners'   : list of 4×2 corner arrays in pixel coords
      - 'scores'    : list of confidence scores
      - 'n_detected': int
    """
    img_bgr = cv2.imread(str(image_path))
    if img_bgr is None:
        raise FileNotFoundError(f"Cannot read: {image_path}")

    results = model.predict(img_bgr, conf=conf, verbose=False)
    result  = results[0]

    crops   = []
    corners_list = []
    scores  = []

    if result.obb is not None and len(result.obb) > 0:
        for i in range(len(result.obb)):
            # xyxyxyxy: shape (1, 4, 2) — 4 corners in pixel coords
            corners_px = result.obb.xyxyxyxy[i].cpu().numpy()  # (4, 2)
            score      = float(result.obb.conf[i].cpu())
            crop       = perspective_warp(img_bgr, corners_px)
            crops.append(crop)
            corners_list.append(corners_px)
            scores.append(score)

    return {
        "original":   img_bgr,
        "crops":      crops,
        "corners":    corners_list,
        "scores":     scores,
        "n_detected": len(crops),
    }


print("OBB helpers defined.")

### A3 — Visualise detections on test images

In [ ]:
def draw_obb(img_bgr: np.ndarray, corners_list: list, scores: list) -> np.ndarray:
    """Draw OBB polygons + confidence scores on a copy of the image."""
    vis = img_bgr.copy()
    for corners, score in zip(corners_list, scores):
        pts = order_corners(corners).astype(np.int32)
        cv2.polylines(vis, [pts], isClosed=True, color=(0, 255, 255), thickness=3)
        cv2.putText(vis, f"{score:.2f}",
                    tuple(pts[0]),  # TL corner
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 255), 2)
    return vis


def visualise_detection(image_path: str | Path, conf: float = CONF_THRESHOLD):
    """Detect, draw OBBs, show original + all warped crops side-by-side."""
    det = detect_and_warp(image_path, conf)
    n   = det["n_detected"]

    print(f"{'─'*60}")
    print(f"Image: {Path(image_path).name}  |  Detected: {n} card(s)")

    n_cols = 1 + n   # original + one per crop
    fig, axes = plt.subplots(1, max(n_cols, 2), figsize=(5 * max(n_cols, 2), 6))

    # Original with OBB overlay
    vis = draw_obb(det["original"], det["corners"], det["scores"])
    axes[0].imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    axes[0].set_title(f"Detection ({n} found)", fontsize=10)
    axes[0].axis("off")

    # Warped crops
    for i, crop in enumerate(det["crops"]):
        axes[i + 1].imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
        axes[i + 1].set_title(f"Crop {i}  conf={det['scores'][i]:.2f}", fontsize=10)
        axes[i + 1].axis("off")

    # Hide unused axes
    for j in range(n + 1, len(axes)):
        axes[j].axis("off")

    plt.tight_layout()
    plt.show()
    return det


# Run on all test images
all_detections = {}
for img_path in TEST_IMAGES:
    if Path(img_path).exists():
        all_detections[img_path] = visualise_detection(img_path)
    else:
        print(f"⚠️  Not found: {img_path}")

### A4 — Save warped crops to disk

In [ ]:
OUT_CROPS_DIR = Path("crops_obb")
OUT_CROPS_DIR.mkdir(exist_ok=True)

saved = []
for img_path, det in all_detections.items():
    stem = Path(img_path).stem
    for i, crop in enumerate(det["crops"]):
        out_path = OUT_CROPS_DIR / f"{stem}_card{i}.jpg"
        cv2.imwrite(str(out_path), crop, [cv2.IMWRITE_JPEG_QUALITY, 95])
        saved.append(out_path)
        print(f"  Saved: {out_path}")

print(f"\nTotal crops saved: {len(saved)}")

### A5 — Feed crops into our existing centering detector

Demonstrates the full pipeline: OBB extract → warp → centering analysis.

In [ ]:
# ── Centering helpers (Python port of our cvDetectors.ts logic) ───────────────
def detect_card_bounds_cv(img_bgr: np.ndarray) -> Optional[dict]:
    """
    Find the outer card boundary in a crop using Sobel gradient.
    Returns {x, y, w, h} or None.
    """
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    h, w = gray.shape
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    sobel_x = cv2.Sobel(blur, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(blur, cv2.CV_64F, 0, 1, ksize=3)
    mag = np.sqrt(sobel_x**2 + sobel_y**2)

    # Column profile for left/right edges
    col_profile = mag.mean(axis=0)
    threshold = col_profile.max() * 0.3
    left  = next((i for i in range(w) if col_profile[i] > threshold), 0)
    right = next((i for i in range(w-1, -1, -1) if col_profile[i] > threshold), w-1)

    # Row profile for top/bottom edges
    row_profile = mag.mean(axis=1)
    top    = next((i for i in range(h) if row_profile[i] > threshold), 0)
    bottom = next((i for i in range(h-1, -1, -1) if row_profile[i] > threshold), h-1)

    if right <= left or bottom <= top:
        return None
    return {"x": left, "y": top, "w": right - left, "h": bottom - top}


def detect_inner_frame_cv(card_bgr: np.ndarray) -> Optional[dict]:
    """
    Scan inward from each edge using a Sobel gradient profile to find the
    inner printed frame boundary. Returns {x, y, w, h, confidence} or None.
    """
    gray = cv2.cvtColor(card_bgr, cv2.COLOR_BGR2GRAY)
    h, w = gray.shape
    blur = cv2.GaussianBlur(gray, (3, 3), 0)
    sobel_x = np.abs(cv2.Sobel(blur, cv2.CV_64F, 1, 0, ksize=3))
    sobel_y = np.abs(cv2.Sobel(blur, cv2.CV_64F, 0, 1, ksize=3))

    SCAN_FRAC = 0.30
    PEAK_RATIO = 0.25

    def find_edge_from(profile, peak_ratio=PEAK_RATIO):
        threshold = profile.max() * peak_ratio
        idx = np.where(profile > threshold)[0]
        return int(idx[0]) if len(idx) > 0 else None

    left_scan  = int(w * SCAN_FRAC)
    right_scan = int(w * SCAN_FRAC)
    top_scan   = int(h * SCAN_FRAC)
    bot_scan   = int(h * SCAN_FRAC)

    col_l = sobel_x[:, :left_scan].mean(axis=0)
    col_r = sobel_x[:, w-right_scan:].mean(axis=0)[::-1]
    row_t = sobel_y[:top_scan, :].mean(axis=1)
    row_b = sobel_y[h-bot_scan:, :].mean(axis=1)[::-1]

    frame_left   = find_edge_from(col_l)
    frame_right  = find_edge_from(col_r)
    frame_top    = find_edge_from(row_t)
    frame_bottom = find_edge_from(row_b)

    if any(v is None for v in [frame_left, frame_right, frame_top, frame_bottom]):
        return None

    frame_x  = frame_left
    frame_y  = frame_top
    frame_w  = w - frame_left - frame_right
    frame_h  = h - frame_top  - frame_bottom

    if frame_w <= 0 or frame_h <= 0:
        return None

    margin_symmetry = 1 - abs(frame_left - frame_right) / max(frame_left + frame_right, 1)
    confidence = "high" if margin_symmetry > 0.7 else "medium" if margin_symmetry > 0.4 else "low"

    return {"x": frame_x, "y": frame_y, "w": frame_w, "h": frame_h,
            "confidence": confidence,
            "margins": {"left": frame_left, "right": frame_right,
                        "top": frame_top, "bottom": frame_bottom}}


def build_centering(card_w, card_h, inner) -> dict:
    """
    Compute centering ratios from outer card dims + inner frame margins.
    All margins expressed as % of card dimension (sum to 100).
    """
    m = inner["margins"]
    total_h = m["left"] + m["right"]
    total_v = m["top"]  + m["bottom"]

    if total_h == 0 or total_v == 0:
        return {}

    left_pct   = round(m["left"]   / total_h * 100, 1)
    right_pct  = round(m["right"]  / total_h * 100, 1)
    top_pct    = round(m["top"]    / total_v * 100, 1)
    bottom_pct = round(m["bottom"] / total_v * 100, 1)

    lr_ratio = max(left_pct, right_pct) / min(left_pct, right_pct) if min(left_pct, right_pct) > 0 else 99
    tb_ratio = max(top_pct, bottom_pct) / min(top_pct, bottom_pct) if min(top_pct, bottom_pct) > 0 else 99

    def interpret(ratio):
        if ratio <= 1.1:  return "well_centered"
        if ratio <= 1.35: return "slightly_off"
        if ratio <= 1.65: return "noticeably_off"
        return "severely_off"

    return {
        "left": left_pct, "right": right_pct,
        "top": top_pct,   "bottom": bottom_pct,
        "lr_ratio": round(lr_ratio, 2), "tb_ratio": round(tb_ratio, 2),
        "lr_interp": interpret(lr_ratio),
        "tb_interp": interpret(tb_ratio),
    }


def plot_centering_on_crop(crop_bgr: np.ndarray, title: str = ""):
    """Visualise outer border + inner frame + centering ratios on a warped crop."""
    h, w = crop_bgr.shape[:2]

    inner = detect_inner_frame_cv(crop_bgr)

    fig, ax = plt.subplots(figsize=(5, 7))
    ax.imshow(cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB))

    # Outer card boundary (blue)
    outer_rect = mpatches.Rectangle((0, 0), w-1, h-1,
                                    linewidth=2, edgecolor="royalblue",
                                    facecolor="none", linestyle="--")
    ax.add_patch(outer_rect)

    centering = {}
    if inner:
        # Inner frame (yellow)
        inner_rect = mpatches.Rectangle(
            (inner["x"], inner["y"]), inner["w"], inner["h"],
            linewidth=2, edgecolor="gold", facecolor="none"
        )
        ax.add_patch(inner_rect)
        centering = build_centering(w, h, inner)

        if centering:
            ax.text(w/2, h + 20,
                    f"L/R: {centering['left']:.0f}/{centering['right']:.0f}  "
                    f"T/B: {centering['top']:.0f}/{centering['bottom']:.0f}  "
                    f"[{centering['lr_interp']}]",
                    ha="center", va="top", fontsize=8,
                    transform=ax.transData)

    ax.set_title(f"{title}\nInner frame: {'found' if inner else 'not found'}", fontsize=9)
    ax.axis("off")
    plt.tight_layout()
    plt.show()
    return centering


# Run centering on all saved crops
print("Centering Analysis on OBB Crops")
print("═" * 60)
for crop_path in saved:
    crop_bgr = cv2.imread(str(crop_path))
    if crop_bgr is None:
        continue
    c = plot_centering_on_crop(crop_bgr, title=crop_path.stem)
    if c:
        print(f"  {crop_path.stem}: L/R {c['left']:.0f}/{c['right']:.0f} ({c['lr_interp']}) "
              f"| T/B {c['top']:.0f}/{c['bottom']:.0f} ({c['tb_interp']})")

---
## Part B — Custom Training Dataset
### B1 — Gather source card images

In [ ]:
# Collect all card images from our datasets folder
IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp"}

card_files = [
    p for p in Path("datasets").rglob("*")
    if p.suffix.lower() in IMG_EXTS
    # Exclude dataset output dir and synthetic images
    and "yolo_obb_cards" not in str(p)
    and "backgrounds" not in str(p)
]
random.shuffle(card_files)
print(f"Found {len(card_files)} card images across datasets/")

# Preview sample
sample = random.sample(card_files, min(6, len(card_files)))
fig, axes = plt.subplots(1, len(sample), figsize=(3 * len(sample), 4))
for ax, p in zip(axes, sample):
    try:
        img = Image.open(p).convert("RGB")
        img.thumbnail((200, 280))
        ax.imshow(img)
        ax.set_title(p.parent.name[:15], fontsize=7)
    except Exception:
        ax.set_title("err")
    ax.axis("off")
plt.suptitle(f"Sample card images ({len(card_files)} total)", fontsize=10)
plt.tight_layout()
plt.show()

### B2 — Generate background images

If you don't have a `datasets/backgrounds/` folder, this cell creates simple
synthetic backgrounds (solid colours, gradients, noise textures).
For better results: add real photos of tables, hands, shelves, etc.

In [ ]:
BACKGROUNDS_DIR.mkdir(parents=True, exist_ok=True)

existing_bgs = list(BACKGROUNDS_DIR.glob("*.*"))
print(f"Background images already present: {len(existing_bgs)}")

if len(existing_bgs) < 20:
    print("Generating synthetic backgrounds...")
    N_SYNTHETIC_BGS = 50
    BG_SIZE = (640, 640)

    for i in range(N_SYNTHETIC_BGS):
        kind = i % 4
        arr = np.zeros((*BG_SIZE, 3), dtype=np.uint8)

        if kind == 0:   # solid colour
            color = [random.randint(30, 230) for _ in range(3)]
            arr[:] = color

        elif kind == 1:  # gradient
            c1 = np.array([random.randint(20, 200) for _ in range(3)])
            c2 = np.array([random.randint(20, 200) for _ in range(3)])
            for row in range(BG_SIZE[0]):
                arr[row] = (c1 + (c2 - c1) * row / BG_SIZE[0]).astype(np.uint8)

        elif kind == 2:  # noise
            arr = np.random.randint(100, 180, (*BG_SIZE, 3), dtype=np.uint8)
            arr = cv2.GaussianBlur(arr, (21, 21), 0)

        else:            # wood-like texture (horizontal streaks)
            base_color = np.array([random.randint(80, 160),
                                   random.randint(50, 100),
                                   random.randint(20, 60)])
            for row in range(BG_SIZE[0]):
                noise = np.random.randint(-20, 20, 3)
                arr[row] = np.clip(base_color + noise, 0, 255).astype(np.uint8)

        out = BACKGROUNDS_DIR / f"bg_{i:04d}.jpg"
        cv2.imwrite(str(out), arr, [cv2.IMWRITE_JPEG_QUALITY, 85])

    print(f"Generated {N_SYNTHETIC_BGS} synthetic backgrounds → {BACKGROUNDS_DIR}")
    print("Tip: add real table/hand photos to this folder for better generalisation.")

bg_files = list(BACKGROUNDS_DIR.glob("*.*"))
print(f"Total backgrounds: {len(bg_files)}")

### B3 — Synthetic composite generator

Adapted directly from `trainCreator.py` (1vcian/Pokemon-TCGP-Card-Scanner).
Key changes for our use case:
- Works with JPEG/WebP card images (not only PNG with alpha)
- Larger scale range (10–85%) to cover eBay-style single-card shots
- Extra placement mode: single large card (eBay listing simulation)
- White border synthesis to simulate PSA/raw card white edges

In [ ]:
# ── IoU helper ────────────────────────────────────────────────────────────────
def iou(b1, b2):
    x1, y1 = max(b1[0], b2[0]), max(b1[1], b2[1])
    x2, y2 = min(b1[2], b2[2]), min(b1[3], b2[3])
    inter = max(0, x2-x1) * max(0, y2-y1)
    u = (b1[2]-b1[0])*(b1[3]-b1[1]) + (b2[2]-b2[0])*(b2[3]-b2[1]) - inter
    return inter / u if u > 0 else 0


# ── Single card placement ─────────────────────────────────────────────────────
def place_card(card_pil: Image.Image, bg_pil: Image.Image,
               existing_boxes: list, scale: Optional[float] = None,
               max_attempts: int = 50):
    """
    Composite `card_pil` onto `bg_pil` at a random position.
    Returns (composited_image, yolo_obb_label_list, pixel_bbox) or None on failure.

    YOLO OBB label format: [class, x1,y1, x2,y2, x3,y3, x4,y4]  (0-1 normalised)
    """
    bg_w, bg_h = bg_pil.size
    card_w, card_h = card_pil.size

    if scale is None:
        scale = random.uniform(0.10, 0.85)

    new_w = int(bg_w * scale)
    new_h = int(card_h * new_w / card_w)
    orig_w, orig_h = new_w, new_h
    card_resized = card_pil.resize((new_w, new_h), Image.LANCZOS)

    # Rotation: ±5° for large cards, ±20° for small
    max_angle = 5 if scale > 0.4 else 20
    angle = random.uniform(-max_angle, max_angle)
    card_rotated = card_resized.rotate(angle, expand=True, resample=Image.BICUBIC)
    rot_w, rot_h = card_rotated.size

    paste_x, paste_y = 0, 0
    for _ in range(max_attempts):
        px = random.randint(0, max(0, bg_w - rot_w))
        py = random.randint(0, max(0, bg_h - rot_h))
        x1, y1 = px, py
        x2, y2 = px + rot_w, py + rot_h
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(bg_w, x2), min(bg_h, y2)
        if (x2 - x1) < 20 or (y2 - y1) < 20:
            continue
        pixel_bbox = [x1, y1, x2, y2]
        if any(iou(pixel_bbox, eb) > 0.1 for eb in existing_boxes):
            continue
        paste_x, paste_y = px, py
        break
    else:
        return None   # couldn't place without overlap

    # Composite
    result = bg_pil.copy()
    if card_rotated.mode == "RGBA":
        mask = card_rotated.split()[3]
        result.paste(card_rotated.convert("RGB"), (paste_x, paste_y), mask)
    else:
        result.paste(card_rotated, (paste_x, paste_y))

    # OBB corner calculation (rotate unrotated corners around centre)
    cx = paste_x + rot_w / 2
    cy = paste_y + rot_h / 2
    hw, hh = orig_w / 2, orig_h / 2
    raw_corners = [(-hw, -hh), (hw, -hh), (hw, hh), (-hw, hh)]
    rad = math.radians(-angle)   # PIL rotates CCW → negate
    obb_coords = [0]  # class 0 = pokemon_card
    for (rx, ry) in raw_corners:
        xr = rx * math.cos(rad) - ry * math.sin(rad)
        yr = rx * math.sin(rad) + ry * math.cos(rad)
        obb_coords.append(min(1, max(0, (cx + xr) / bg_w)))
        obb_coords.append(min(1, max(0, (cy + yr) / bg_h)))

    return result, obb_coords, pixel_bbox


print("Composite helpers defined.")

### B4 — Dataset generator loop

In [ ]:
def generate_dataset(
    card_files: list,
    bg_files:   list,
    out_dir:    Path,
    n_images:   int = N_TRAIN_IMAGES,
    img_size:   int = 640,
    train_frac: float = 0.8,
    val_frac:   float = 0.1,
) -> Path:
    """
    Generate a YOLO OBB dataset with `n_images` synthetic composites.

    Three scene modes (sampled randomly):
      'single_large'  : 1 card at 50–85% width  (eBay-style clean listing)
      'few_cards'     : 2–4 cards at 20–40% width
      'many_cards'    : 5–15 cards at 8–18% width

    Returns path to data.yaml.
    """
    splits = {
        "train": int(n_images * train_frac),
        "val":   int(n_images * val_frac),
        "test":  n_images - int(n_images * train_frac) - int(n_images * val_frac),
    }

    for split in splits:
        (out_dir / "images" / split).mkdir(parents=True, exist_ok=True)
        (out_dir / "labels" / split).mkdir(parents=True, exist_ok=True)

    idx = 0
    for split, count in splits.items():
        imgs_dir   = out_dir / "images" / split
        labels_dir = out_dir / "labels" / split

        for _ in tqdm(range(count), desc=f"Generating {split}"):
            bg_path = random.choice(bg_files)
            bg_pil  = Image.open(bg_path).convert("RGB")
            bg_pil  = bg_pil.resize((img_size, img_size), Image.LANCZOS)

            # Choose scene mode
            mode = random.choices(
                ["single_large", "few_cards", "many_cards"],
                weights=[0.40, 0.35, 0.25]
            )[0]

            if mode == "single_large":
                n_cards = 1
                scale_range = (0.50, 0.85)
            elif mode == "few_cards":
                n_cards = random.randint(2, 4)
                scale_range = (0.20, 0.40)
            else:
                n_cards = random.randint(5, 15)
                scale_range = (0.08, 0.18)

            existing_boxes = []
            labels = []
            composite = bg_pil

            for _ in range(n_cards):
                card_path = random.choice(card_files)
                try:
                    card_pil = Image.open(card_path).convert("RGBA")
                except Exception:
                    continue
                scale = random.uniform(*scale_range)
                result = place_card(card_pil, composite, existing_boxes, scale)
                if result is None:
                    continue
                composite, obb_label, pixel_bbox = result
                existing_boxes.append(pixel_bbox)
                labels.append(" ".join(f"{v:.6f}" if i > 0 else str(int(v))
                                       for i, v in enumerate(obb_label)))

            if not labels:
                continue   # skip empty images

            img_name   = f"img_{idx:07d}.jpg"
            label_name = f"img_{idx:07d}.txt"
            composite.save(imgs_dir / img_name, quality=90)
            (labels_dir / label_name).write_text("\n".join(labels))
            idx += 1

    # Write data.yaml
    yaml_data = {
        "path":   str(out_dir.resolve()),
        "train":  "images/train",
        "val":    "images/val",
        "test":   "images/test",
        "nc":     1,
        "names":  ["pokemon_card"],
    }
    yaml_path = out_dir / "data.yaml"
    yaml_path.write_text(yaml.dump(yaml_data, default_flow_style=False))
    print(f"\n✅  Dataset generated: {idx} images → {out_dir}")
    print(f"   data.yaml: {yaml_path}")
    return yaml_path


# ── Run dataset generation ─────────────────────────────────────────────────────
# Set N_TRAIN_IMAGES in the config cell. Start small (200–500) to verify
# quality, then increase to 5000–10000 for production training.
yaml_path = generate_dataset(card_files, bg_files, DATASET_OUT, n_images=N_TRAIN_IMAGES)

### B5 — Preview synthetic composites

In [ ]:
# Show 6 random generated training images with OBB labels drawn
train_imgs = list((DATASET_OUT / "images" / "train").glob("*.jpg"))
sample_imgs = random.sample(train_imgs, min(6, len(train_imgs)))

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for ax, img_path in zip(axes, sample_imgs):
    img_bgr = cv2.imread(str(img_path))
    label_path = DATASET_OUT / "labels" / "train" / (img_path.stem + ".txt")

    if label_path.exists():
        h, w = img_bgr.shape[:2]
        for line in label_path.read_text().strip().split("\n"):
            vals = list(map(float, line.split()))
            # vals[1:] = x1,y1,x2,y2,x3,y3,x4,y4 normalised
            pts = np.array(vals[1:]).reshape(4, 2)
            pts[:, 0] *= w
            pts[:, 1] *= h
            pts = pts.astype(np.int32)
            cv2.polylines(img_bgr, [pts], isClosed=True,
                          color=(0, 255, 255), thickness=2)

    ax.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    ax.set_title(img_path.stem, fontsize=8)
    ax.axis("off")

plt.suptitle("Sample synthetic training images with OBB labels", fontsize=12)
plt.tight_layout()
plt.show()

### B6 — Train YOLO11n-obb on generated dataset

In [ ]:
# ── Training config ───────────────────────────────────────────────────────────
EPOCHS      = 50      # increase to 100-150 for full training
IMG_SZ      = 640
BATCH       = 8       # reduce to 4 if OOM on MPS
DEVICE      = "mps"   # Apple Silicon; use 'cuda' for NVIDIA, 'cpu' as fallback

# ─────────────────────────────────────────────────────────────────────────────
print(f"Training YOLO11n-obb for {EPOCHS} epochs on {yaml_path}")
print(f"Device: {DEVICE}  |  Batch: {BATCH}  |  ImgSz: {IMG_SZ}")

train_model = YOLO(PRETRAINED_BASE)   # YOLO11n-obb as starting point

results = train_model.train(
    data    = str(yaml_path),
    epochs  = EPOCHS,
    imgsz   = IMG_SZ,
    batch   = BATCH,
    device  = DEVICE,
    project = str(DATASET_OUT),
    name    = "train",
    exist_ok= True,
    patience= 20,        # early stop if no improvement for 20 epochs
    save    = True,
    plots   = True,
)

best_weights = DATASET_OUT / "train" / "weights" / "best.pt"
print(f"\nTraining complete. Best weights: {best_weights}")

### B7 — Evaluate custom model & compare with base

In [ ]:
if best_weights.exists():
    custom_model = YOLO(str(best_weights))

    print("Custom model — test on real card photos")
    print("═" * 60)
    for img_path in TEST_IMAGES:
        if not Path(img_path).exists():
            continue
        img_bgr = cv2.imread(img_path)
        results = custom_model.predict(img_bgr, conf=CONF_THRESHOLD, verbose=False)
        n_det   = len(results[0].obb) if results[0].obb is not None else 0

        fig, ax = plt.subplots(figsize=(6, 8))
        vis = img_bgr.copy()
        if results[0].obb is not None:
            for i in range(n_det):
                corners = results[0].obb.xyxyxyxy[i].cpu().numpy()
                score   = float(results[0].obb.conf[i].cpu())
                pts     = order_corners(corners).astype(np.int32)
                cv2.polylines(vis, [pts], True, (0, 255, 0), 3)
                cv2.putText(vis, f"{score:.2f}", tuple(pts[0]),
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
        ax.set_title(f"{Path(img_path).name}  |  {n_det} detected", fontsize=9)
        ax.axis("off")
        plt.tight_layout()
        plt.show()
else:
    print("No trained weights found — run B6 first.")

### B8 — Full pipeline: noisy photo → warped crop → centering

In [ ]:
# ── Config ─────────────────────────────────────────────────────────────────────
PIPELINE_IMAGE = "cards/image0_front.jpeg"   # ← any photo with a Pokemon card
# ──────────────────────────────────────────────────────────────────────────────

# Use best available model
pipeline_model = YOLO(str(best_weights)) if best_weights.exists() else model

img_bgr  = cv2.imread(PIPELINE_IMAGE)
results  = pipeline_model.predict(img_bgr, conf=CONF_THRESHOLD, verbose=False)
obb_res  = results[0].obb

if obb_res is None or len(obb_res) == 0:
    print("No cards detected. Try lowering CONF_THRESHOLD or use a trained model.")
else:
    # Take highest-confidence detection
    best_i   = int(obb_res.conf.argmax())
    corners  = obb_res.xyxyxyxy[best_i].cpu().numpy()
    score    = float(obb_res.conf[best_i].cpu())
    crop_bgr = perspective_warp(img_bgr, corners)

    # Centering on warped crop
    inner    = detect_inner_frame_cv(crop_bgr)
    centering = build_centering(CARD_W, CARD_H, inner) if inner else {}

    # Visualise 3-stage pipeline
    fig, axes = plt.subplots(1, 3, figsize=(16, 6))

    # Stage 1: detection
    vis = img_bgr.copy()
    pts = order_corners(corners).astype(np.int32)
    cv2.polylines(vis, [pts], True, (0, 255, 255), 3)
    axes[0].imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    axes[0].set_title(f"1. OBB Detection (conf={score:.2f})", fontsize=10)
    axes[0].axis("off")

    # Stage 2: warped crop
    axes[1].imshow(cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB))
    axes[1].set_title(f"2. Perspective Warp ({CARD_W}×{CARD_H}px)", fontsize=10)
    axes[1].axis("off")

    # Stage 3: centering overlay
    crop_vis = crop_bgr.copy()
    h, w = crop_vis.shape[:2]
    if inner:
        # Draw outer (blue dashed) + inner (yellow)
        cv2.rectangle(crop_vis, (0,0), (w-1,h-1), (255, 80, 0), 2)
        cv2.rectangle(crop_vis,
                      (inner["x"], inner["y"]),
                      (inner["x"]+inner["w"], inner["y"]+inner["h"]),
                      (0, 215, 255), 2)
    axes[2].imshow(cv2.cvtColor(crop_vis, cv2.COLOR_BGR2RGB))
    if centering:
        title = (f"3. Centering\n"
                 f"L/R {centering['left']:.0f}/{centering['right']:.0f} "
                 f"({centering['lr_interp']})\n"
                 f"T/B {centering['top']:.0f}/{centering['bottom']:.0f} "
                 f"({centering['tb_interp']})")
    else:
        title = "3. Centering\n(inner frame not found)"
    axes[2].set_title(title, fontsize=10)
    axes[2].axis("off")

    plt.suptitle("Full Pipeline: Photo → OBB → Warp → Centering", fontsize=13)
    plt.tight_layout()
    plt.show()

    print("\nCentering result:")
    for k, v in centering.items():
        print(f"  {k}: {v}")